# Tasks 2 & 3 — Feature Engineering, Model Development & MLflow Tracking
**Heart Disease UCI Dataset**  
MLOps Assignment 01 | AIMLCZG523

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve, auc
from data_processing import load_processed, prepare_train_test, ALL_FEATURES
from train import train_and_log, tune_random_forest, save_best_model

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='whitegrid')
os.makedirs('../models', exist_ok=True)

## 2.1 Load Processed Data

In [ ]:
df = load_processed('../data/processed/heart_disease_clean.csv')
X_train, X_test, y_train, y_test = prepare_train_test(df)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Train class balance: {y_train.value_counts().to_dict()}')
print(f'Test  class balance: {y_test.value_counts().to_dict()}')

## 2.2 Feature Scaling Overview
We use a `StandardScaler` applied inside the pipeline to prevent data leakage.  
Categorical features are kept as numeric codes (already integer-encoded in UCI format).

In [ ]:
from data_processing import build_preprocessing_pipeline, CONTINUOUS_FEATURES, CATEGORICAL_FEATURES
print('Continuous features (scaled):', CONTINUOUS_FEATURES)
print('Categorical features (imputed, no scaling):', CATEGORICAL_FEATURES)
print('Total features:', len(ALL_FEATURES))

## 2.3 MLflow Setup

In [ ]:
MLFLOW_URI = '../mlruns'
EXPERIMENT = 'heart-disease-classification'
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)
print(f'MLflow tracking URI: {mlflow.get_tracking_uri()}')
print(f'Experiment: {EXPERIMENT}')

## 2.4 Model 1 — Logistic Regression
Baseline model. Interpretable, fast, good for binary classification.

In [ ]:
lr_params = {'C': 1.0, 'solver': 'lbfgs', 'max_iter': 1000, 'random_state': 42}
lr_clf = LogisticRegression(**lr_params)
lr_pipeline, lr_metrics = train_and_log(
    'Logistic Regression', lr_clf, lr_params, X_train, X_test, y_train, y_test
)
pd.DataFrame([lr_metrics]).T.rename(columns={0: 'Score'})

## 2.5 Model 2 — Random Forest (with Hyperparameter Tuning)
Ensemble model. Robust to outliers, captures non-linear relationships.

In [ ]:
best_rf_params, _ = tune_random_forest(X_train, y_train)
rf_clf = RandomForestClassifier(
    n_estimators=best_rf_params['classifier__n_estimators'],
    max_depth=best_rf_params['classifier__max_depth'],
    min_samples_split=best_rf_params['classifier__min_samples_split'],
    random_state=42
)
rf_plain = {k.replace('classifier__', ''): v for k, v in best_rf_params.items()}
rf_pipeline, rf_metrics = train_and_log(
    'Random Forest', rf_clf, rf_plain, X_train, X_test, y_train, y_test
)
pd.DataFrame([rf_metrics]).T.rename(columns={0: 'Score'})

## 2.6 Model 3 — Gradient Boosting
Sequential ensemble that corrects previous errors. Often top performer.

In [ ]:
gb_params = {'n_estimators': 150, 'learning_rate': 0.1, 'max_depth': 4, 'random_state': 42}
gb_clf = GradientBoostingClassifier(**gb_params)
gb_pipeline, gb_metrics = train_and_log(
    'Gradient Boosting', gb_clf, gb_params, X_train, X_test, y_train, y_test
)
pd.DataFrame([gb_metrics]).T.rename(columns={0: 'Score'})

## 2.7 Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Logistic Regression': lr_metrics,
    'Random Forest': rf_metrics,
    'Gradient Boosting': gb_metrics,
}).T
comparison = comparison[['accuracy', 'precision', 'recall', 'f1', 'roc_auc',
                          'cv_accuracy_mean', 'cv_roc_auc_mean']]
comparison.style.highlight_max(color='#d5f5e3', axis=0).format('{:.4f}')

In [ ]:
# Bar chart comparison
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(metrics_to_plot))
width = 0.25
models_data = {
    'Logistic Regression': lr_metrics,
    'Random Forest': rf_metrics,
    'Gradient Boosting': gb_metrics,
}
colors = ['#3498db', '#2ecc71', '#e74c3c']
for i, (name, metrics) in enumerate(models_data.items()):
    vals = [metrics[m] for m in metrics_to_plot]
    bars = ax.bar(x + i*width, vals, width, label=name, color=colors[i], edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../screenshots/model_comparison.png')
plt.show()

In [ ]:
# ROC curves overlay
fig, ax = plt.subplots(figsize=(7, 6))
for (name, pipeline), color in zip(
    [('Logistic Regression', lr_pipeline), ('Random Forest', rf_pipeline), ('Gradient Boosting', gb_pipeline)],
    colors
):
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC={roc_auc:.3f})')

ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../screenshots/roc_all_models.png')
plt.show()

## 2.8 Select & Save Best Model

In [ ]:
results = {
    'Logistic Regression': (lr_pipeline, lr_metrics),
    'Random Forest': (rf_pipeline, rf_metrics),
    'Gradient Boosting': (gb_pipeline, gb_metrics),
}
best_name = max(results, key=lambda k: results[k][1]['roc_auc'])
best_pipeline, best_metrics = results[best_name]
print(f'Best model: {best_name}')
print(f'ROC-AUC  : {best_metrics["roc_auc"]}')
save_best_model(best_pipeline, 'best_model')

## 2.9 MLflow Experiment Summary

In [ ]:
client = mlflow.tracking.MlflowClient(tracking_uri=MLFLOW_URI)
exp = client.get_experiment_by_name(EXPERIMENT)
runs = client.search_runs(exp.experiment_id, order_by=['metrics.roc_auc DESC'])
run_data = [{
    'Model': r.data.tags.get('model_name', r.info.run_name),
    'Accuracy': r.data.metrics.get('accuracy'),
    'ROC-AUC': r.data.metrics.get('roc_auc'),
    'F1': r.data.metrics.get('f1'),
    'Run ID': r.info.run_id[:8],
} for r in runs]
pd.DataFrame(run_data)